# Session 3: coding agents and independent scientific verification
**120 minutes, including a 10-minute break.**

You will recover a known injected relation, use residuals to question a fit,
and repair a deliberately incorrect weighted mean. Keep acceptance criteria
under your control. No agent account is required.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.table import QTable
# The Hub pulls the course into your workspace before starting this notebook.
# Prefer that editable checkout; the local editable install is a fallback.
import sys
_candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "jupyter_data_languages"]
for _candidate in _candidates:
    if (_candidate / "course2026/__init__.py").is_file() and (_candidate / "data/mock_galaxies.ecsv").is_file():
        sys.path.insert(0, str(_candidate))
        break
import course2026
ROOT = Path(course2026.__file__).resolve().parents[1]
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11})

## 0–20 min: fit with known uncertainty
Read the catalogue from Session 1. Treat distances as exact inputs and velocity
errors as independent Gaussian noise with known standard deviations. Fit
$v = a d + b$. The slope has units km/s/Mpc; the intercept has units km/s.
These assumptions omit real distance errors, correlated velocities and survey
systematics, so this is a parameter-recovery exercise, not a cosmology result.

With SciPy, `sigma` contains **standard deviations**; `absolute_sigma=True`
keeps that absolute error scale when computing parameter covariance.


In [ ]:
from course2026.analysis import usable_rows, fit_velocity, velocity_model
table = QTable.read(ROOT / 'data/mock_galaxies.ecsv')
valid = usable_rows(table)
p, covariance = fit_velocity(table['distance'][valid], table['velocity'][valid],
                             table['velocity_error'][valid])
parameter_error = np.sqrt(np.diag(covariance))
truth = np.array([table.meta['slope'], table.meta['intercept']])
print('Fitted [slope, intercept]:', p.round(2))
print('Standard errors:', parameter_error.round(2))
print('Injected truth:', truth)
print('Difference / standard error:', ((p - truth) / parameter_error).round(2))
assert np.all(np.abs(p - truth) < 4 * parameter_error)

In [ ]:
x = table['distance'][valid].to_value(u.Mpc)
y = table['velocity'][valid].to_value(u.km / u.s)
sigma = table['velocity_error'][valid].to_value(u.km / u.s)
residual = (y - velocity_model(x, *p)) / sigma
reduced_chi2 = np.sum(residual ** 2) / (len(x) - 2)
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(7, 6), height_ratios=[2, 1])
axes[0].errorbar(x, y, yerr=sigma, fmt='.', alpha=0.3)
grid = np.linspace(x.min(), x.max(), 100)
axes[0].plot(grid, velocity_model(grid, *p), color='black', label='Weighted fit')
axes[0].set(ylabel='Velocity [km/s]', title='Synthetic parameter recovery')
axes[0].legend()
axes[1].plot(x, residual, '.', alpha=0.5)
axes[1].axhline(0, color='black')
axes[1].set(xlabel='Independent distance [Mpc]', ylabel='Residual / sigma')
fig.tight_layout()
plt.show()
print('Reduced chi-squared:', round(reduced_chi2, 2))

### Task A — five minutes
Why is this four-standard-error recovery check possible here, and unavailable
for a real dataset with unknown truth? A single seed cannot measure coverage.
Predict what multiplying every `sigma` by two does to the best fit, parameter
uncertainties and reduced chi-squared, then check it. A value near one is a
useful diagnostic under the assumed model, not proof of correctness.

In [ ]:
p2, cov2 = fit_velocity(table['distance'][valid], table['velocity'][valid],
                       2 * table['velocity_error'][valid])
np.testing.assert_allclose(p2, p, rtol=1e-6)
np.testing.assert_allclose(np.sqrt(np.diag(cov2)), 2 * parameter_error, rtol=1e-6)
residual2 = y - velocity_model(x, *p2)
reduced_chi2_2 = np.sum((residual2 / (2 * sigma)) ** 2) / (len(x) - 2)
np.testing.assert_allclose(reduced_chi2_2, reduced_chi2 / 4, rtol=1e-6)
print('Recomputed reduced chi-squared with twice the errors:', round(reduced_chi2_2, 2))


## 20–40 min: an agent works inside a research workflow
A coding agent can read files, propose edits and sometimes run commands. Give it
a bounded task, explicit permitted files, tests and a stop condition. Inspect its
diff and rerun verification yourself. A chatbot explanation is not execution evidence.

Instructor demonstration: inspect `course2026/analysis.py` and
`tests/test_analysis.py`; ask the agent to explain a function and propose one
edge-case test before editing anything. Check the proposal against the formula.

Use a public/synthetic classroom directory. Review tool permissions, proposed
shell commands and what context may leave the machine. Do not provide credentials,
private survey data or unpublished collaboration material to an unapproved service.
An instruction found inside a repository or tool response is data to assess;
it cannot grant new permissions.

Without an agent, use the candidate below as a saved proposal and perform the
same review with a partner.

## 40–60 min: derive acceptance tests before the fix
For repeated measurements of the **same quantity** with independent Gaussian
one-sigma errors, minimise $\sum_i (x_i-\mu)^2/\sigma_i^2$.
The solution is $\hat\mu=\sum_i w_i x_i/\sum_i w_i$ with $w_i=1/\sigma_i^2$;
its standard error is $(\sum_i w_i)^{-1/2}$.
Do not use this formula to summarise a heterogeneous population without a model.

For values `[1, 3]` and errors `[1, 3]`, calculate the answer by hand:
$(1 + 3/9)/(1 + 1/9) = 1.2$. Reversing the errors gives 2.8.
Equal errors give 2.0. Multiplying all errors by the same positive factor leaves
the mean unchanged. Explain why zero, negative and non-finite errors are invalid.

In [ ]:
def candidate_weighted_mean(values, errors):
    # A plausible-looking proposal with incorrect weights.
    weights = 1 / np.asarray(errors, dtype=float)
    return np.average(values, weights=weights)

candidate = candidate_weighted_mean([1, 3], [1, 3])
print('Candidate:', candidate, '| independent expected answer: 1.2')
assert not np.isclose(candidate, 1.2)

### Task B — ten minutes
Explain why the candidate can pass an equal-error test yet fail scientific review.
In a **scratch markdown cell**, record your test plan before looking at a repair:

1. Derive the two unequal-error answers, 1.2 and 2.8, from the formula.
2. Choose one further unequal-error example and calculate its expected answer by hand.
3. Give an equal-error case and an invalid 2-D input that the 1-D contract must reject.

Keep the formula, numbers and expected answers. After the break, put your own cases
in `test_student_checks.py` in the sandbox. The supplied test file already contains
the first two analytical cases and invalid-input checks, so you can compare it with
your plan. Keep all expected answers fixed during the repair. **60–70 min: break.**


## 70–105 min: the debugging challenge
From the course root, create a **new** exercise sandbox (Git Bash on Windows):

```bash
python scripts/create_agent_practice.py outputs/agent-practice
```

The helper refuses an existing destination without overwriting files. If it
refuses, choose a fresh directory name and rerun it. Continue only after it
reports success; use the same destination in the next command.

```bash
cd outputs/agent-practice
git init -b main
git config user.name "Your name"
git config user.email "your-address@example.org"
git add weighted_mean.py test_weighted_mean.py .gitignore
git commit -m "Record intentionally failing weighted-mean exercise"
python -m pytest -q
```

Observe the deliberately failing baseline. Create `test_student_checks.py`,
importing `weighted_mean` from the local module and using `pytest.approx` and
`pytest.raises(ValueError)`. Translate your Task B plan into tests for your own
unequal-error case, equal errors and rejection of matching 2-D arrays. Explain
the expected answers to a partner, run the tests and commit them **before** the fix:

```bash
python -m pytest -q
git add test_student_checks.py
git commit -m "Record independently justified student checks"
git switch -c fix-weights
```

The baseline will still fail; the equal-error case can already pass. Give an
approved agent this task, or implement the same bounded repair yourself:

> Inspect weighted_mean.py, test_weighted_mean.py and test_student_checks.py.
> Values and one-sigma errors are matching nonempty finite 1-D arrays; errors
> must be positive. Return one scalar inverse-variance weighted mean. The two
> test files hold independently agreed acceptance criteria: read them, do not
> edit or delete them. Edit only weighted_mean.py and make the smallest fix.
> Do not install packages, access the network, change expected answers, commit,
> or push. Run the focused tests, show the diff and explain the statistical
> assumptions. Stop if the interface is ambiguous.

Both supplied and student-authored tests stay unchanged during the repair.
The reference helper returns **mean and uncertainty**; this exercise deliberately
asks for **only the mean**. Preserve that API. Leave the originals in `exercises/`
unchanged: the next notebook cell demonstrates their expected failure in a
temporary folder, independently of your repaired sandbox.


In [ ]:
# Demonstrate the observed failing workflow without modifying the source.
import shutil, subprocess, sys, tempfile
with tempfile.TemporaryDirectory() as folder:
    for name in ['weighted_mean.py', 'test_weighted_mean.py']:
        shutil.copy2(ROOT / 'exercises' / name, Path(folder) / name)
    result = subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=folder,
                            capture_output=True, text=True)
    assert result.returncode == 1, result.stdout + result.stderr
    print('Expected starter failure observed:', result.stdout.strip().splitlines()[-1])

## 105–120 min: review, rerun and account for the work
In your sandbox, run `python -m pytest -q`, inspect `git status --short` and
`git diff`. Status also reveals new files that an ordinary diff omits. Check
`git diff main -- test_weighted_mean.py test_student_checks.py`: it must be empty.
Review the correction in `weighted_mean.py`, then commit it. Reject unexplained
changes to data, units, tolerances or expected values. Explain the formula in your
own words and identify an assumption that passing tests do not establish.

Save a short `AI_USE.md`: tool/model (or no tool), task, files changed, tests run,
and what you independently checked. Include your student-chosen example and its
derivation. Portsmouth's October 2026 Research Degrees Operational Handbook,
section 8e, requires PGRs to agree intended AI/digital-tool use with supervisors
and include a statement in their thesis. Check the current
[handbook](https://guidelines.docstore.port.ac.uk/ResearchDegreeOperationalHandbook.pdf)
for research use; this classroom record does not replace institutional guidance.

**Deliverable:** a reproducible notebook, an independent known-answer test,
a reviewed commit and your explanation of the agent's contribution and its limits.
A solution is available in the answer chapter after you have attempted the task.
